In [1]:
import fasttext
import iso639
from huggingface_hub import hf_hub_download

/home/m/cvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_path = hf_hub_download(repo_id="HPLT/OpenLID-v3", filename="openlid-v3.bin")
model = fasttext.load_model(model_path)
ol_labels = model.get_labels()

model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
model = fasttext.load_model(model_path)
gl_labels = model.get_labels()

In [70]:
ol_labels.sort()

In [3]:
assert "__label__dyu_Latn" in gl_labels

In [5]:
assert "__label__bam_Latn" in gl_labels

In [7]:
assert "__label__twi_Latn" in ol_labels

In [8]:
assert "__label__aka_Latn" in ol_labels

AssertionError: 

In [9]:
assert "__label__fat_Latn" in ol_labels

AssertionError: 

In [14]:
assert "__label__twi_Latn" in gl_labels

In [15]:
assert "__label__aka_Latn" in gl_labels

AssertionError: 

In [71]:
gl_labels.sort()

In [23]:
assert "__label__pes_Arab" in gl_labels

AssertionError: 

In [16]:
assert "__label__fat_Latn" in gl_labels

In [7]:
"__label__fat_Latn".split("_")

['', '', 'label', '', 'fat', 'Latn']

In [8]:
iso639.Language.from_part3('ara')

Language(part3='ara', part2b='ara', part2t='ara', part1='ar', scope='M', type='L', status='A', name='Arabic', comment=None, other_names=None, macrolanguage=None, retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [24]:
iso639.Language.from_part3('bam')

Language(part3='bam', part2b='bam', part2t='bam', part1='bm', scope='I', type='L', status='A', name='Bambara', comment=None, other_names=None, macrolanguage=None, retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [25]:
iso639.Language.from_part3('dyu')

Language(part3='dyu', part2b='dyu', part2t='dyu', part1=None, scope='I', type='L', status='A', name='Dyula', comment=None, other_names=None, macrolanguage=None, retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [3]:
iso639.Language.from_part3('arb')

Language(part3='arb', part2b=None, part2t=None, part1=None, scope='I', type='L', status='A', name='Standard Arabic', comment=None, other_names=[Name(print='Standard Arabic', inverted='Arabic, Standard')], macrolanguage='ara', retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [28]:
iso639.Language.from_part3('ara')

Language(part3='ara', part2b='ara', part2t='ara', part1='ar', scope='M', type='L', status='A', name='Arabic', comment=None, other_names=None, macrolanguage=None, retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [14]:
from collections import defaultdict

In [72]:
def make_macro2ind(labels):
    macro2sep = defaultdict(list)
    for label in labels:
        part3 = label.split("_")[4]
        try:
            lang = iso639.Language.from_part3(part3)
        except iso639.LanguageNotFoundError as e:
            print(e)
            continue
        if lang.macrolanguage is not None:
            # print(label)
            # print(lang.name)
            # print(lang.macrolanguage)
            # print(iso639.Language.from_part3(lang.macrolanguage).name)
            macro2sep[lang.macrolanguage].append(label[-8:])
        elif lang.scope == "M":
            macro2sep[part3].append(label)
    return macro2sep

In [73]:
macro2ind_gl = make_macro2ind(gl_labels)

'nah' isn't an ISO language code or name
'oto' isn't an ISO language code or name


In [67]:
def compare(labels_to_compare, macro2sep, model_name="glotlid", model_to_compare="OpenLID"):
    labels_pure = [label.removeprefix("__label__") for label in labels_to_compare]
    labels_no_script = [label[:3] for label in labels_pure]
    with open(f"labels/macro2ind.{model_name}", "w") as f:
        for macro, separates in macro2sep.items():
            f.write(f"{macro}, {iso639.Language.from_part3(macro).name}\n")
            ind_codes = [part3[-8:-5] for part3 in separates]
            if macro in labels_no_script:
                f.write(
                    f"{model_to_compare} uses {macro} for {separates} ({[iso639.Language.from_part3(part3).name for part3 in ind_codes]})\n"
                )
            if macro in ind_codes:
                if macro in labels_no_script:
                    f.write(f"{model_name} also uses macrolanguage here!\n")
                else:
                    f.write(f"Only {model_name} has this macrolanguage (as macrolanguage)\n")
            else:
                for sep in separates:
                    name = iso639.Language.from_part3(sep[-8:-5]).name
                    if sep in labels_pure:
                        f.write(f"{model_to_compare} uses {sep} ({name})\n")
                    else:
                        f.write(f"{model_to_compare} has no {sep} ({name})\n")
            f.write('---------------------------------\n')

In [68]:
compare(ol_labels, macro2ind_gl, model_name="glotlid", model_to_compare="OpenLID")

In [21]:
iso639.Language.from_part3("fas")

Language(part3='fas', part2b='per', part2t='fas', part1='fa', scope='M', type='L', status='A', name='Persian', comment=None, other_names=None, macrolanguage=None, retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [22]:
iso639.Language.from_part3("pes")

Language(part3='pes', part2b=None, part2t=None, part1=None, scope='I', type='L', status='A', name='Iranian Persian', comment=None, other_names=[Name(print='Iranian Persian', inverted='Persian, Iranian')], macrolanguage='fas', retire_reason=None, retire_change_to=None, retire_remedy=None, retire_date=None)

In [74]:
macro2ind_ol = make_macro2ind(ol_labels)
compare(gl_labels, macro2ind_ol, model_name="openlid", model_to_compare="GlotLID")